In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Evaluation Code with Multiple Metrics
-------------------------------------
본 코드는 다음 지표들을 계산합니다.

1) CoverageFraction(avg, per file)
   - (파일별 coverageFraction)의 평균값
   - coverageFraction = (매칭된 GT subtask 수) / (GT subtask 총 수)
   - "bigger is better"

2) Global Coverage Fraction
   - (모든 파일에서 매칭된 GT subtask 총합) / (모든 파일의 GT subtask 총합)
   - "bigger is better"

3) MatchedMakespan(avg, per file)
   - 매칭된 subtask들의 실행 구간을 병합(merge)하여 얻은 길이의, 파일별 평균
   - "smaller is better"

4) MatchedMakespan per Coverage
   - (모든 파일에서의 matched makespan 총합) / (매칭된 subtask 총 수)
   - "smaller is better"

5) MatchedMakespan per GT
   - (모든 파일에서의 matched makespan 총합) / (GT subtask 총 수)
   - "smaller is better"

6) result = avg_matched_makespan / global_coverage_fraction
   - (파일별 평균 matched makespan)을 (전역 coverage fraction)으로 나눈 값
   - coverage가 높을수록 이 값이 낮아지는 형태의 종합 지표
"""

import json
from pathlib import Path
from typing import List, Tuple, Set, Dict

import networkx as nx
from sentence_transformers import SentenceTransformer, util

# 경로 관련 상수 (사용하시는 환경에 맞춰 수정하세요)
from utils.config.constants import RESULT_PATH, TASK_PATH

# 문장 임베딩 모델
model = SentenceTransformer("all-MiniLM-L6-v2")


def build_gt_dict() -> Dict[str, List[str]]:
    """
    TASK_PATH 폴더 내부의 *.json 파일을 전부 읽어,
    "파일이름.json": [GT 서브태스크1, ...], ... 형태의 딕셔너리를 구성합니다.
    """
    gt_dict = {}
    for file in TASK_PATH.iterdir():
        if file.is_file() and file.suffix == ".json":
            fn = file.name
            subtask_list = []
            with file.open("r", encoding="utf-8") as f:
                data = json.load(f)
                for item in data:
                    if "Subtasks" in item:
                        for sb in item["Subtasks"]:
                            if "Name" in sb:
                                subtask_list.append(sb["Name"])
            gt_dict[fn] = subtask_list
    return gt_dict


def multi_match_flow(
    recognized_names: List[str], gt_names: List[str], threshold: float
) -> Tuple[Set[int], Set[int], List[Tuple[int, int]]]:
    """
    recognized_names vs. gt_names를 Flow(최대유량)을 이용해 매핑합니다.

    - Recognized 쪽 노드(각 subtask) capacity = G (GT 수) -> 하나의 Recognized subtask가 여러 GT에 매칭(1:N) 가능
    - GT 쪽 노드(각 subtask) capacity = 1            -> 하나의 GT는 최대 한 Recognized subtask와만 매칭 가능
    - cosine similarity가 threshold 이상일 때만 간선(capacity=1) 연결

    Returns:
        leftover_rec : 매칭 안 된 recognized index 세트
        leftover_gt  : 매칭 안 된 GT index 세트
        matched_pairs: (rec_idx, gt_idx) 형태로 매칭된 쌍들의 리스트
    """
    R = len(recognized_names)
    G = len(gt_names)

    # 둘 중 하나라도 비어 있으면 곧바로 종료
    if R == 0 or G == 0:
        return set(range(R)), set(range(G)), []

    # 문장 임베딩 & 유사도 계산
    rec_embs = model.encode(recognized_names, convert_to_tensor=True)
    gt_embs = model.encode(gt_names, convert_to_tensor=True)
    sim_mat = util.cos_sim(rec_embs, gt_embs).cpu().numpy()

    # Flow 네트워크 구성
    SRC = "SRC"
    SNK = "SNK"
    flow_graph = nx.DiGraph()

    flow_graph.add_node(SRC)
    flow_graph.add_node(SNK)

    # Recognized subtask 쪽 노드 추가 (capacity = G)
    capacity_rec = G if G > 0 else 1
    for i in range(R):
        rec_node = f"REC_{i}"
        flow_graph.add_node(rec_node)
        flow_graph.add_edge(SRC, rec_node, capacity=capacity_rec)

    # GT subtask 쪽 노드 추가 (capacity = 1)
    for j in range(G):
        gt_node = f"GT_{j}"
        flow_graph.add_node(gt_node)
        flow_graph.add_edge(gt_node, SNK, capacity=1)

    # threshold 이상 유사도인 경우 edge(capacity=1) 추가
    for i in range(R):
        for j in range(G):
            if sim_mat[i, j] >= threshold:
                rec_node = f"REC_{i}"
                gt_node = f"GT_{j}"
                flow_graph.add_edge(rec_node, gt_node, capacity=1)

    # Maximum Flow 계산
    flow_val, flow_dict = nx.maximum_flow(flow_graph, SRC, SNK)

    # 매칭된 (i, j) 페어 탐색
    matched_pairs = []
    for i in range(R):
        rec_node = f"REC_{i}"
        outflow = flow_dict.get(rec_node, {})
        for j in range(G):
            gt_node = f"GT_{j}"
            if outflow.get(gt_node, 0) > 0.9999:
                matched_pairs.append((i, j))

    # 매칭된 Recognized, GT index
    used_rec = {x[0] for x in matched_pairs}
    used_gt = {x[1] for x in matched_pairs}

    # leftover(매칭 안 된) Recognized, GT index
    leftover_rec = set(range(R)) - used_rec
    leftover_gt = set(range(G)) - used_gt
    return leftover_rec, leftover_gt, matched_pairs


def merge_intervals(intervals: List[Tuple[float, float]]) -> List[Tuple[float, float]]:
    """
    (startTime, endTime) 구간 리스트에서 겹치는 구간을 병합하여 반환합니다.
    """
    if not intervals:
        return []
    # 시작 시간 기준 정렬 후 병합
    intervals.sort(key=lambda x: x[0])
    merged = []
    current_start, current_end = intervals[0]

    for i in range(1, len(intervals)):
        s, e = intervals[i]
        # 구간이 겹치는지 확인
        if s <= current_end:
            current_end = max(current_end, e)
        else:
            merged.append((current_start, current_end))
            current_start, current_end = s, e
    merged.append((current_start, current_end))
    return merged


def get_file_metrics(
    dag_data: dict, gt_subtask_names: List[str], threshold: float
) -> Tuple[int, float, float, List[str], List[str]]:
    """
    dag_data의 plans[0].subtasks 정보를 이용하여 다음을 계산:
      - coverage_count: 매칭된 subtask 수
      - coverage_fraction: coverage_count / GT subtask 수
      - matched_makespan: 매칭된 subtask들의 실행 구간을 병합한 길이
      - leftover_rec_names, leftover_gt_names: 매칭 안 된 subtask의 이름 목록

    Returns:
      (coverage_count,
       coverage_fraction,
       matched_makespan,
       leftover_rec_names,
       leftover_gt_names)
    """
    # 실제 실행(성공)된 subtask만 필터링
    sub_exec = dag_data["plans"][0]["subtasks"]
    gt_count = len(gt_subtask_names)

    if gt_count == 0:
        # GT가 없는 경우
        recognized_names = [s["subtaskName"] for s in sub_exec]
        return 0, 0.0, 0.0, recognized_names, []

    # executionStatus=True인 subtasks만 대상으로 매칭
    success_indices = []
    success_names = []
    for i, s in enumerate(sub_exec):
        if s.get("executionStatus") is True:
            success_indices.append(i)
            success_names.append(s["subtaskName"])

    # 실행 성공 subtask가 없으면 coverage는 0
    if not success_indices:
        recognized_names = [s["subtaskName"] for s in sub_exec]
        return 0, 0.0, 0.0, recognized_names, gt_subtask_names

    # (A) 매칭 수행 (flow 기반)
    leftover_rec_inds, leftover_gt_inds, matched_succ = multi_match_flow(
        success_names, gt_subtask_names, threshold
    )

    coverage_count = len(matched_succ)  # 매칭된 (rec, gt) 쌍 -> 곧 GT 매칭 수
    coverage_fraction = coverage_count / gt_count if gt_count > 0 else 0.0

    # leftover 인덱스를 이름으로 변환
    leftover_rec_names = [success_names[i] for i in sorted(leftover_rec_inds)]
    leftover_gt_names = [gt_subtask_names[j] for j in sorted(leftover_gt_inds)]

    # (B) 매칭된 subtask들의 (startTime, endTime)를 병합 후 길이 계산
    matched_indices = {success_indices[r_idx] for (r_idx, _) in matched_succ}
    intervals = []
    for idx in matched_indices:
        sdata = sub_exec[idx]
        st = sdata.get("startTime", 0.0) or 0.0
        ed = sdata.get("endTime", st)
        if ed < st:
            ed = st  # 혹시 endTime < startTime인 경우 보정
        intervals.append((st, ed))

    merged_intervals = merge_intervals(intervals)
    matched_makespan = sum(e - s for s, e in merged_intervals)

    return (
        coverage_count,
        coverage_fraction,
        matched_makespan,
        leftover_rec_names,
        leftover_gt_names,
    )


def process_time_folders(
    task_folder: Path, gt_sub_names: List[str], threshold: float
) -> Tuple[int, int, float, float, int]:
    """
    하나의 태스크 폴더(예: TASK.json)에 대해,
    내부의 여러 "타임스탬프 폴더"를 순회하며 metrics 합산:

    Returns (해당 task_folder 기준):
      - coverage_count 합계
      - GT subtask 수 합계
      - matched makespan 합계
      - coverage fraction(파일 단위) 합
      - 처리된 파일(스탬프) 수
    """
    sum_cov_count = 0
    sum_gt_count = 0
    sum_makespan = 0.0
    sum_covFrac = 0.0
    file_count = 0

    # 타임스탬프 폴더 순회
    for stamp_folder in task_folder.iterdir():
        if not stamp_folder.is_dir():
            continue

        # 접근할 JSON 파일 경로 예시
        dag_file = stamp_folder / "approach" / "dag_bayesian.json"
        if not dag_file.exists():
            continue

        with dag_file.open("r", encoding="utf-8") as f:
            dag_data = json.load(f)

        cov_count, cov_frac, matched_ms, leftover_rec, leftover_gt = get_file_metrics(
            dag_data, gt_sub_names, threshold
        )

        # leftover verbose
        if leftover_rec or leftover_gt:
            print(f"[Leftover] {task_folder.name}/{stamp_folder.name}")
            if leftover_rec:
                print(f"  - Unmatched recognized subtasks: {leftover_rec}")
            if leftover_gt:
                print(f"  - Unmatched GT subtasks        : {leftover_gt}")
            print()

        # 누적 계산
        file_count += 1
        sum_cov_count += cov_count
        sum_covFrac += cov_frac
        sum_makespan += matched_ms
        sum_gt_count += len(gt_sub_names)

    return sum_cov_count, sum_gt_count, sum_makespan, sum_covFrac, file_count


def main(folder_name: str, gt_dict: Dict[str, List[str]], threshold: float) -> None:
    """
    folder_name 폴더 내부의 각 "태스크 폴더"(ex. TASK.json 폴더)를 순회하며,
    평가 지표를 계산하고 결과를 출력합니다.
    """
    target_folder = RESULT_PATH / folder_name
    if not target_folder.exists():
        print(f"[ERROR] folder not exist: {target_folder}")
        return

    # 전역(=모든 태스크) 누적값
    grand_cov_count = 0  # 모든 매칭된 GT 수의 합
    grand_gt_count = 0  # 모든 GT 수의 합
    grand_makespan = 0.0  # 모든 매칭된 makespan 합
    grand_covFracSum = 0.0  # 모든 파일의 coverage fraction 합
    grand_file_count = 0  # 처리된 파일(스탬프) 총 수
    task_count = 0  # 유효 태스크 폴더 수

    # target_folder 내에 있는 각 폴더가 GT 파일명과 동일할 경우 처리
    for item in target_folder.iterdir():
        if item.is_dir() and item.name in gt_dict:
            gt_sub_names = gt_dict[item.name]

            (sum_cov, sum_gt, sum_ms, sum_cf, file_cnt) = process_time_folders(
                item, gt_sub_names, threshold
            )

            if file_cnt > 0:  # 처리된 스탬프가 있다면
                grand_cov_count += sum_cov
                grand_gt_count += sum_gt
                grand_makespan += sum_ms
                grand_covFracSum += sum_cf
                grand_file_count += file_cnt
                task_count += 1

    # 최종 결과 계산
    if task_count == 0:
        print(f"\n[{folder_name}] - No data processed.")
        return

    # 1) CoverageFraction(avg, per file)
    avg_file_covFrac = (
        grand_covFracSum / grand_file_count if grand_file_count > 0 else 0.0
    )

    # 2) Global Coverage Fraction
    global_covFrac = (grand_cov_count / grand_gt_count) if grand_gt_count > 0 else 0.0

    # 3) MatchedMakespan(avg, per file)
    avg_matched_makespan = (
        grand_makespan / grand_file_count if grand_file_count > 0 else 0.0
    )

    # 4) Makespan per Coverage
    makespan_per_coverage = (
        grand_makespan / grand_cov_count if grand_cov_count > 0 else 0.0
    )

    # 5) Makespan per GT
    makespan_per_gt = grand_makespan / grand_gt_count if grand_gt_count > 0 else 0.0

    # 6) result = avg_matched_makespan / global_covFrac
    result_metric_6 = (
        avg_matched_makespan / global_covFrac if global_covFrac > 1e-12 else 0.0
    )

    print(f"\n[{folder_name}] (files={grand_file_count}, threshold={threshold})")
    print(
        f"  1) CoverageFraction(avg, per file) : {avg_file_covFrac:.4f} (bigger is better)"
    )
    print(
        f"  2) Global Coverage Fraction        : {global_covFrac:.4f} (bigger is better)"
    )
    print(
        f"  3) MatchedMakespan(avg, per file)  : {avg_matched_makespan:.4f} (smaller is better)"
    )
    print(
        f"  4) Makespan per Coverage           : {makespan_per_coverage:.4f} (smaller is better)"
    )
    print(
        f"  5) Makespan per GT                 : {makespan_per_gt:.4f} (smaller is better)"
    )
    print(f"  6) result = avg_ms / global_covFrac: {result_metric_6:.4f}")


if __name__ == "__main__":
    # GT 정보 사전 구성
    gt_dict = build_gt_dict()
    threshold = 0.8

    folder_names = ["jcci_top1", "jcci_top25", "jcci_zeroshot"]
    for fd in folder_names:
        main(fd, gt_dict, threshold)

[ERROR] folder not exist: /home/dongkyu/pdk_ws/research/assets/results/jcci_top1
[ERROR] folder not exist: /home/dongkyu/pdk_ws/research/assets/results/jcci_top25
[ERROR] folder not exist: /home/dongkyu/pdk_ws/research/assets/results/jcci_zeroshot


[Leftover] complex9_17subtasks(dc2, dnc1, dnc2, dnc3, nd(1, 2, 3)).json/2025-03-20_03_47_03_cook egg fry an.json
  - Unmatched recognized subtasks: ['Navigate to StoveKnob|-00.02|+00.88|-02.19 during 2.4000000000000004', 'Monitoring for Turn off Stove after cooking_12d97283']
  - Unmatched GT subtasks        : ['Wash Lettuce', 'Wash Tomato']

[Leftover] complex9_17subtasks(dc2, dnc1, dnc2, dnc3, nd(1, 2, 3)).json/2025-03-20_03_37_51_cook egg fry an.json
  - Unmatched recognized subtasks: ['Navigate to StoveKnob|-00.02|+00.88|-02.19 during 2.4000000000000004', 'Monitoring for Turn off Stove after cooking_b22c2ed8', 'Wash Knife']
  - Unmatched GT subtasks        : ['Wash Lettuce', 'Wash Tomato']

[Leftover] complex9_17subtasks(dc2, dnc1, dnc2, dnc3, nd(1, 2, 3)).json/2025-03-20_03_42_38_cook egg fry an.json
  - Unmatched recognized subtasks: ['Navigate to StoveKnob|-00.02|+00.88|-02.19 during 2.4000000000000004', 'Monitoring for Turn off Stove after cooking_148fa161']
  - Unmatched GT su

In [4]:
!echo $PWD

/home/dongkyu/pdk_ws/research/src
